In [35]:
import os
import yaml
from dotenv import set_key, load_dotenv


load_dotenv(".env")

repo_code = os.getenv("PCLOUD_CODE")
pcloud_username = os.getenv("PCLOUD_USERNAME")
pcloud_password = os.getenv("PCLOUD_PASSWORD")

In [36]:
from lcdb.db import PCloudRepository
repo = PCloudRepository(repo_code=repo_code)
repo.authenticate(username=pcloud_username, password=pcloud_password, authexpire=86400*2)

dotenv_path = ".env"  
set_key(dotenv_path, "PCLOUD_TOKEN", repo.token)


repo code: kZeWywZRr6lScWSloHlzwk6Uxq3GyRtuBaX


(True, 'PCLOUD_TOKEN', 'hUVUoXZoxkjZUr3qypFMWFFAsVcUJC35Du1MNElk')

In [37]:
repo_token = os.getenv("PCLOUD_TOKEN")

In [38]:
repo = PCloudRepository(repo_code=repo_code, token=repo_token)

repo code: kZeWywZRr6lScWSloHlzwk6Uxq3GyRtuBaX


In [39]:
workflow_mapping = {
  "libsvm": "lcdb.workflow.sklearn.LibSVMWorkflow",
  "randomforest": "lcdb.workflow.sklearn.RandomForestWorkflow",
  "knn": "lcdb.workflow.sklearn.KNNWorkflow",
  "xgboost": "lcdb.workflow.xgboost.XGBoostWorkflow",
  "treesensemble": "lcdb.workflow.sklearn.TreesEnsembleWorkflow",
  "liblinear": "lcdb.workflow.sklearn.LibLinearWorkflow"
}

In [40]:
# Load the YAML configuration from a file
with open("experiments/surf/snellius/scripts/config.yaml", "r") as file:
    config = yaml.safe_load(file)

# Print the parsed configuration
print(config)
# get necessary parameters from the configuration file
workflow = config["workflow_name"]
workflow_class = workflow_mapping[workflow]
memory = config["desired_memory_GB"]
test_seeds = config["test_seeds"]
val_seeds = config["val_seeds"]

campaing_name = f"data_probing-{memory}"

{'workflow_name': 'liblinear', 'desired_memory_GB': 67, 'config_num': 20, 'workflow_seed': 42, 'val_seeds': [0], 'test_seeds': [0]}


In [41]:
def get_folder_id(workflow, memory):
  return repo._get_folder_id(f"data/{workflow}/data_probing-{memory}")

folder_id = get_folder_id(workflow_class, memory)

In [42]:
import requests

response = requests.get(
    f"https://eapi.pcloud.com/listfolder?code={repo.repo_code}&auth={repo.token}&folderid=14175375152"
).json()
print(response)

{'result': 0, 'metadata': {'name': 'lcdb', 'created': 'Thu, 05 Dec 2024 09:11:31 +0000', 'ismine': True, 'thumb': False, 'modified': 'Thu, 05 Dec 2024 09:11:39 +0000', 'comments': 0, 'id': 'd14175375152', 'isshared': False, 'icon': 'folder', 'isfolder': True, 'parentfolderid': 14175372679, 'folderid': 14175375152, 'contents': [{'name': 'data', 'created': 'Thu, 05 Dec 2024 09:11:39 +0000', 'ismine': True, 'thumb': False, 'modified': 'Mon, 03 Mar 2025 13:34:47 +0000', 'comments': 0, 'id': 'd14175377067', 'isshared': False, 'icon': 'folder', 'isfolder': True, 'parentfolderid': 14175375152, 'folderid': 14175377067}]}}


In [43]:
repo.content

{'result': 0,
 'metadata': {'name': 'data_probing-67',
  'created': 'Wed, 26 Mar 2025 05:44:08 +0000',
  'ismine': True,
  'thumb': False,
  'modified': 'Wed, 26 Mar 2025 05:48:17 +0000',
  'comments': 0,
  'id': 'd16017264874',
  'isshared': False,
  'icon': 'folder',
  'isfolder': True,
  'parentfolderid': 14175848526,
  'folderid': 16017264874,
  'contents': [{'name': '1067',
    'created': 'Wed, 26 Mar 2025 05:44:34 +0000',
    'ismine': True,
    'thumb': False,
    'modified': 'Wed, 26 Mar 2025 05:44:34 +0000',
    'comments': 0,
    'id': 'd16017269608',
    'isshared': False,
    'icon': 'folder',
    'isfolder': True,
    'parentfolderid': 16017264874,
    'folderid': 16017269608},
   {'name': '1111',
    'created': 'Wed, 26 Mar 2025 05:44:38 +0000',
    'ismine': True,
    'thumb': False,
    'modified': 'Wed, 26 Mar 2025 05:44:38 +0000',
    'comments': 0,
    'id': 'd16017270110',
    'isshared': False,
    'icon': 'folder',
    'isfolder': True,
    'parentfolderid': 16017

In [44]:
import requests
import csv

def get_subfolder_names(auth_token, repo_code, folder_id):
    url = "https://eapi.pcloud.com/listfolder"
    params = {
        "code": repo_code,
        "auth": auth_token,
        "folderid": folder_id
    }

    response = requests.get(url, params=params).json()

    if response.get("result") == 0:
        try:
            # Extract and sort folder names numerically
            folders = sorted(
                (item["name"] for item in response["metadata"].get("contents", []) if item.get("isfolder")),
                key=int  # Convert to int for correct numerical sorting
            )
            return folders
        except ValueError:
            print("Warning: Some folder names are not purely numeric. Falling back to default sorting.")
            return sorted(
                (item["name"] for item in response["metadata"].get("contents", []) if item.get("isfolder"))
            )
    else:
        print(f"Error: {response.get('error', 'Unknown error')}")
        return []

def save_to_csv(folders, filename="folders.csv"):
    with open(filename, mode="w", newline="") as file:
        writer = csv.writer(file)
        # writer.writerow(["Folder Name"])  # Add header row
        for folder in folders:
            writer.writerow([folder])
    print(f"Saved {len(folders)} folders to {filename}")

# Example usage
auth_token = repo.token
repo_code = repo.repo_code

folders = get_subfolder_names(auth_token, repo_code, folder_id)

if folders:
    save_to_csv(folders, filename=f"liblinear-{memory}-datasets.csv")
    print(f"Subfolders: {folders}")


Saved 67 folders to liblinear-67-datasets.csv
Subfolders: ['3', '12', '23', '31', '54', '181', '188', '1067', '1111', '1169', '1457', '1461', '1464', '1468', '1475', '1486', '1487', '1489', '1494', '1515', '1590', '1596', '4134', '4135', '4534', '4538', '4541', '23517', '40668', '40670', '40685', '40701', '40900', '40975', '40978', '40981', '40982', '40983', '40996', '41027', '41138', '41142', '41143', '41144', '41145', '41146', '41147', '41150', '41156', '41157', '41158', '41159', '41161', '41163', '41164', '41165', '41166', '41167', '41168', '41169', '42732', '42733', '42734', '42742', '42746', '42769', '43072']
